In [ ]:
import re
import html
import unicodedata

import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt


# Fermented Food Inventory

In [ ]:
food_inventory_df = pd.read_csv("../data/indonesian_fermented_foods.csv")
print(f"Food Inventory DF Shape: {food_inventory_df.shape}")

Food Inventory DF Shape: (85, 6)


In [7]:
display(food_inventory_df)

,fermented_food,raw_material,fermentation_process,microorganism_present,references,regions
0,Sayur asin,Mustard cabbage leaf,Lactic fermentation,"Lactobacillus farciminis, Lb. fermentum, Lb. n...",10.1016/j.fshw.2015.06.001,NaN
1,Tempoyak,Flesh of durian (Durio zibethinus),Lactic fermentation,"Lb. plantarum, Lb. coryneformis, Lb. casei; Lb...",10.1016/j.fshw.2015.06.001,NaN
2,Mandai,"Flesh of cempedak, a family of jack fruit (Art...",Lactic fermentation,Lb. plantarum; P. pentosaceus,10.1016/j.fshw.2015.06.001,NaN
3,Tape starter culture,Rice flour,–,"P. pentosaceus, E. faecium, Lb. curvatus, W. c...",10.1016/j.fshw.2015.06.001,NaN
4,Rice wine/rice tape,Glutinuous rice (steamed),Alcoholic fermentation,"P. pentosaceus, Weisella sp.",10.1016/j.fshw.2015.06.001,NaN
...,...,...,...,...,...,...
80,Budik,Pork blood,NaN,NaN,10.20961/jaht.v1i2.473,"Rote, Kupang, Timor"
81,Bekasam,"Fish: teri, nila, saluang, wader, sepat siam, ...",Spontaneous anaerobic fermentation with salt a...,Lactobacillus plantarum,10.20961/jaht.v1i2.473,"South Sumatra, South Kalimantan"
82,Naniura,"Common carp (ikan mas), andaliman, jeruk batak...",Raw fish marinated in jeruk jugga or another a...,"Lactobacillus sp. 1, Lactobacillus sp. 2, Lact...",10.20961/jaht.v1i2.473,"Toba, North Sumatra"
83,Bebontot/buntilan,"Pork, chicken, or beef; spices",Natural fermentation without a defined lactic ...,"Lactobacillus plantarum, Pediococcus pentosace...",10.20961/jaht.v1i2.473,Bali


# Get Microorganism

In [ ]:
# Get All Mentioned Microorganism
microorganisms = [
    name.strip()
    for item in food_inventory_df["microorganism_present"].dropna()
    for name in item.split(", ")
    if name.strip()
]
microorganisms = sorted(set(microorganisms))
microorganisms = pd.Series(microorganisms, name="microorganism")
display(microorganisms.head(10))

0                           A. oryzae
1                            A. sojae
2                Acetic acid bacteria
3                   Acetobacter aceti
4                      Actinobacteria
5                      Aerococcus sp.
6                   Amylomyces rouxii
7    Aneurinibacillus aneurinilyticus
8                  Aspergillus flavus
9                   Aspergillus niger
Name: microorganism, dtype: object

# Fermented Food Literature

In [3]:
ff_lit_df = pd.read_csv("../data/fermented_food_literatures.csv")

In [4]:
queries = ff_lit_df['standard_query'].unique()

print("Fermented Food Literature Related")
print(f"Query:")
for query in queries:
    print(query)
print(f"Shape: {ff_lit_df.shape}")
print()

Fermented Food Literature Related
Query:
(("fermented food" OR "fermented foods" OR "traditional fermented food" OR "fermented beverage" OR "pangan fermentasi" OR "makanan fermentasi" OR "minuman fermentasi") AND (Indonesia OR Indonesian))
Shape: (1781, 24)



In [5]:
display(ff_lit_df.head(10))
print()

,doi,pmid,pmcid,title,authors,abstract,publication_year,publication_date,journal,is_open_access,...,europe_pmc_source,publication_type,language,citation_count,url,pdf_url,source_database,query,standard_query,executed_query
0,10.1016/j.copbio.2026.103573,42636734.0,NaN,From traditional fermented food to microbial f...,"Asih SC, Choi KR.","Microbial food, food and ingredients produced ...",2026.0,2026-08-24,Current opinion in biotechnology,False,...,MED,Review; Journal Article,eng,0.0,https://europepmc.org/article/MED/42636734,NaN,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
1,10.1111/1758-2229.70346,42244023.0,PMC13238706,Polyphasic Characterization of Acetobacter ind...,"Syaputri Y, Zahra E, Azzahra RN, Annisa, Iswha...",Acetobacter indonesiensis UNPADCC 01-5 is a re...,2026.0,2026-06-01,Environmental microbiology reports,True,...,MED,research-article; Journal Article,eng,0.0,https://europepmc.org/article/MED/42244023,https://europepmc.org/articles/PMC13238706?pdf...,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
2,10.1093/femsmc/xtag005,41660005.0,PMC12875122,Traditional fermented foods of Indonesia harbo...,"Wicaksono WA, Zukancic E, Zlatnar M, Suwanto A...",Fermented foods represent complex microbial ec...,2026.0,2026-01-22,FEMS microbes,True,...,MED,research-article; Journal Article,eng,1.0,https://europepmc.org/article/MED/41660005,https://europepmc.org/articles/PMC12875122?pdf...,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
3,10.3390/molecules31081353,42076031.0,PMC13118699,Probiotics and Fermented Foods in Human Nutrit...,Dini I.,Fermented foods and probiotics represent compl...,2026.0,2026-04-20,"Molecules (Basel, Switzerland)",True,...,MED,review-article; Review; Journal Article,eng,1.0,https://europepmc.org/article/MED/42076031,https://europepmc.org/articles/PMC13118699?pdf...,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
4,10.1111/1750-3841.71014,41906408.0,NaN,A Critical Review of Antidiabetic Properties o...,"Ayustaningwarno F, Jusman TNG, Afifah DN, Asik...",Tempeh is a traditional fermented food from se...,2026.0,2026-04-01,Journal of food science,False,...,MED,Review; Journal Article,eng,0.0,https://europepmc.org/article/MED/41906408,NaN,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
5,10.3389/fmicb.2026.1860268,42445497.0,PMC13357518,Unlocking the health potential of fermented fo...,"Raali R, Sampath N, R S G, J K T, Vardhan J H,...",Fermented foods have been an integral componen...,2026.0,2026-06-29,Frontiers in microbiology,True,...,MED,review-article; Review; Journal Article,eng,0.0,https://europepmc.org/article/MED/42445497,https://europepmc.org/articles/PMC13357518?pdf...,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr..."
6,10.1590/1519-6984.298140,41849523.0,NaN,"Metagenomic analysis of Gatot, a cassava-based...","Mardiyaningsih A, Astuti P, Widodo W, Purwanto P.","Gatot, a traditional Indonesian fermented food...",2026.0,2026-03-13,Brazilian journal of biology = Revista braslei...,False,...,MED,Journal Article,eng,0.0,https://europepmc.org/article/MED/41849523,NaN,Europe PMC,"((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food"" OR ""fermented foods"" OR ""tr...","((""fermented food""

**Check Missing Value**

In [6]:
count_missing = (
    ff_lit_df.isna()
    .sum()
    .to_frame("n_missing")
    .assign(pct_missing=lambda x: round((x["n_missing"] / len(ff_lit_df)) * 100))
    .assign(data_type=ff_lit_df.dtypes)
    .sort_values("n_missing", ascending=False)
    .reset_index(names="column_name") 
)

count_missing.index = count_missing.index + 1

display(count_missing)

,column_name,n_missing,pct_missing,data_type
1,openalex_id,1781,100.0,float64
2,semantic_scholar_id,1215,68.0,object
3,semantic_scholar_corpus_id,1215,68.0,float64
4,pmcid,892,50.0,object
5,europe_pmc_id,888,50.0,object
6,europe_pmc_source,888,50.0,object
7,pmid,784,44.0,float64
8,pdf_url,589,33.0,object
9,language,572,32.0,object
10,citation_count,322,18.0,float64
